# Practice 3: Customer Clustering - Bước 7: Đánh giá mô hình (Model Evaluation)

---

## 1. Import các thư viện và tải dữ liệu đặc trưng đầy đủ

Chúng ta tải dữ liệu đã tiền xử lý từ `../data/processed_features/`.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import os

%matplotlib inline

# Load dữ liệu 22 chiều đã xử lý đặc trưng
X_train = pd.read_csv('../data/processed_features/X_train.csv').values
X_val = pd.read_csv('../data/processed_features/X_val.values' if os.path.exists('../data/processed_features/X_val.values') else '../data/processed_features/X_val.csv').values

print(f'Kích thước dữ liệu huấn luyện: {X_train.shape}')
print(f'Kích thước dữ liệu xác thực:  {X_val.shape}')

Kích thước dữ liệu huấn luyện: (6455, 26)
Kích thước dữ liệu xác thực:  (1613, 26)


## 2. Huấn luyện các mô hình từ Scikit-learn làm đại diện

Chúng ta chạy hai thuật toán đại diện tiêu biểu nhất từ thư viện `scikit-learn` trên cùng tập Train:
1. **K-Means** với số cụm tối ưu toán học $K = 3$.
2. **Hierarchical (Agglomerative)** với cấu hình tối ưu $K = 4$, liên kết `ward`.

In [7]:
# 2.1. Huấn luyện KMeans từ sklearn
sklearn_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = sklearn_kmeans.fit_predict(X_train)

# 2.2. Huấn luyện Hierarchical từ sklearn
# Lưu ý: Để so sánh đồng nhất với bước trước, ta huấn luyện trên cùng 2000 mẫu ngẫu nhiên đã dùng ở bước 6
np.random.seed(42)
sub_train_idx = np.random.choice(X_train.shape[0], 2000, replace=False)
X_train_sub = X_train[sub_train_idx]

sklearn_hierarchical = AgglomerativeClustering(n_clusters=4, linkage='ward')
hierarchical_labels = sklearn_hierarchical.fit_predict(X_train_sub)

print('Đã huấn luyện xong hai mô hình đại diện từ Scikit-learn!')

Đã huấn luyện xong hai mô hình đại diện từ Scikit-learn!


## 3. Tính toán độ lệch và tương đồng giữa 2 kết quả phân cụm

Để đo lường độ đồng thuận chéo giữa phương pháp phân hoạch (K-Means, K=3) và phương pháp phân cấp (Hierarchical, K=4), chúng ta so sánh nhãn của chúng trên tập 2000 mẫu chung.

In [8]:
# Trích xuất nhãn KMeans tương ứng với 2000 mẫu tập con
kmeans_labels_sub = kmeans_labels[sub_train_idx]

# 3.1. Tính các chỉ số đồng thuận thống kê
ari_score = adjusted_rand_score(kmeans_labels_sub, hierarchical_labels)
nmi_score = normalized_mutual_info_score(kmeans_labels_sub, hierarchical_labels)

print('--- CHỈ SỐ TƯƠNG ĐỒNG GIỮA KMEANS (K=3) VÀ HIERARCHICAL (K=4) ---')
print(f'  - Adjusted Rand Index (ARI): {ari_score:.4f}')
print(f'  - Normalized Mutual Information (NMI): {nmi_score:.4f}')
print('\n-> Nhận xét: ARI đo mức độ đồng thuận ngẫu nhiên, NMI đo lượng thông tin chia sẻ.')
print('Điểm số phản ánh sự chồng lấn trung bình giữa hai cấu hình số cụm khác nhau.')

--- CHỈ SỐ TƯƠNG ĐỒNG GIỮA KMEANS (K=3) VÀ HIERARCHICAL (K=4) ---
  - Adjusted Rand Index (ARI): 0.4039
  - Normalized Mutual Information (NMI): 0.5593

-> Nhận xét: ARI đo mức độ đồng thuận ngẫu nhiên, NMI đo lượng thông tin chia sẻ.
Điểm số phản ánh sự chồng lấn trung bình giữa hai cấu hình số cụm khác nhau.


## 5. Đối chiếu kiểm chứng thuật toán viết từ Scratch

Chúng ta đối chiếu kết quả của mô hình Scikit-learn với kết quả lưu từ mô hình tự viết (Scratch) ở Bước 6 để chứng minh tính chính xác tuyệt đối.

In [20]:
# 5.1. Kiểm chứng K-Means
scratch_km_labels = np.load('../models/kmeans_labels.npy')
km_verify_score = adjusted_rand_score(kmeans_labels, scratch_km_labels)
print(f'Độ trùng khớp ARI giữa KMeans Scratch và KMeans Sklearn: {km_verify_score:.6f}')

# 5.2. Kiểm chứng Hierarchical
with open('../models/hierarchical_scratch_labels.pkl', 'rb') as f:
    scratch_hier_labels = pickle.load(f)
hier_verify_score = adjusted_rand_score(hierarchical_labels, scratch_hier_labels)
print(f'Độ trùng khớp ARI giữa Hierarchical Scratch và Hierarchical Sklearn: {hier_verify_score:.6f}')



Độ trùng khớp ARI giữa KMeans Scratch và KMeans Sklearn: 1.000000
Độ trùng khớp ARI giữa Hierarchical Scratch và Hierarchical Sklearn: 1.000000


## 6. Lưu trữ nhãn phân cụm đại diện phục vụ phân tích chân dung

Chúng ta xuất tệp tin chứa nhãn phân cụm để sử dụng ở các bước phân tích chân dung khách hàng tiếp theo.

In [11]:
os.makedirs('../data/processed_features/', exist_ok=True)

# Lưu nhãn KMeans đầy đủ
pd.DataFrame({'KMeans_Label': kmeans_labels}).to_csv('../data/processed_features/kmeans_labels_sklearn.csv', index=False)

# Lưu nhãn Hierarchical cùng chỉ số tương ứng
pd.DataFrame({
    'Train_Sub_Index': sub_train_idx,
    'Hierarchical_Label': hierarchical_labels
}).to_csv('../data/processed_features/hierarchical_labels_sklearn.csv', index=False)

print('Đã lưu trữ nhãn phân cụm đại diện thành công!')

Đã lưu trữ nhãn phân cụm đại diện thành công!
